## 1. Chargement et préparation de l’image

On commence par charger la galaxie en niveaux de gris, puis on réduit sa taille pour que les calculs soient plus rapides et on normalise les pixels entre 0 et 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

image_path = Path("milky-way-nvg.jpg")
if not image_path.exists():
    raise FileNotFoundError("Place milky-way-nvg.jpg dans le même dossier que le notebook.")

image = Image.open(image_path).convert("L")
image = image.resize((256, 256))
image_array = np.asarray(image, dtype=np.float64) / 255.0
hauteur, largeur = image_array.shape
pixels = image_array.reshape(-1, 1)

plt.figure(figsize=(5, 5))
plt.imshow(image_array, cmap="gray", vmin=0, vmax=1)
plt.title("Image en niveaux de gris")
plt.axis("off")
plt.show()
print("Nombre de pixels :", pixels.shape[0])

## 2. Initialisation

Ensuite, on choisit deux clusters et un coefficient de flou égal à 2, puis on initialise les degrés d’appartenance de chaque pixel en vérifiant que leur somme vaut 1.

In [ ]:
K = 2
m = 2.0
epsilon = 0.001
max_iterations = 100

generateur = np.random.default_rng(42)
U = generateur.random((len(pixels), K))
U = U / U.sum(axis=1, keepdims=True)

print("Taille de U :", U.shape)
print("Exemple de degrés d’appartenance :", U[0])
print("Somme :", U[0].sum())

## 3. Implémentation de Fuzzy C-Means

À chaque itération, on calcule les centres avec les degrés d’appartenance, puis on met à jour ces degrés à partir des distances entre les pixels et les centres. On répète les calculs jusqu’à ce que les valeurs changent très peu ou qu’on atteigne le nombre maximal d’itérations.

In [ ]:
def calculer_centres(X, U, m):
    poids = U ** m
    return (poids.T @ X) / poids.sum(axis=0)[:, None]


def mettre_a_jour_appartenances(X, centres, m):
    distances = np.linalg.norm(X[:, None, :] - centres[None, :, :], axis=2)
    nouvelles_U = np.zeros_like(distances)

    # Si un pixel est exactement sur un centre, on évite une division par zéro.
    sur_centre = distances <= 1e-12
    lignes_exactes = sur_centre.any(axis=1)
    if np.any(lignes_exactes):
        valeurs = sur_centre[lignes_exactes].astype(float)
        nouvelles_U[lignes_exactes] = valeurs / valeurs.sum(axis=1, keepdims=True)

    autres = ~lignes_exactes
    if np.any(autres):
        d = distances[autres]
        inverse = d ** (-2.0 / (m - 1.0))
        nouvelles_U[autres] = inverse / inverse.sum(axis=1, keepdims=True)

    return nouvelles_U


def fuzzy_c_means(X, K=2, m=2.0, epsilon=0.001, max_iterations=100, seed=42):
    if m <= 1:
        raise ValueError("m doit être supérieur à 1.")
    rng = np.random.default_rng(seed)
    U = rng.random((len(X), K))
    U /= U.sum(axis=1, keepdims=True)
    historique = []

    for iteration in range(max_iterations):
        centres = calculer_centres(X, U, m)
        nouvelles_U = mettre_a_jour_appartenances(X, centres, m)
        changement = np.max(np.abs(nouvelles_U - U))
        U = nouvelles_U
        historique.append(changement)
        if changement < epsilon:
            break

    centres = calculer_centres(X, U, m)
    return centres, U, historique

centres, U_final, historique = fuzzy_c_means(
    pixels, K=K, m=m, epsilon=epsilon, max_iterations=max_iterations
)

# On range les deux clusters du plus sombre au plus clair.
ordre = np.argsort(centres[:, 0])
centres = centres[ordre]
U_final = U_final[:, ordre]

print("Centres (sombre, clair) :", centres.ravel())
print("Nombre d’itérations :", len(historique))
print("Dernier changement :", historique[-1])
print("Sommes des appartenances proches de 1 :", np.allclose(U_final.sum(axis=1), 1))

## 4. Segmentation floue

On transforme les degrés d’appartenance en deux images, puis on utilise des heatmaps pour voir quels pixels appartiennent plutôt à la région sombre ou à la région claire

In [ ]:
carte_sombre = U_final[:, 0].reshape(hauteur, largeur)
carte_claire = U_final[:, 1].reshape(hauteur, largeur)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_array, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Image originale")
axes[1].imshow(carte_sombre, cmap="inferno", vmin=0, vmax=1)
axes[1].set_title("Appartenance : région sombre")
im = axes[2].imshow(carte_claire, cmap="inferno", vmin=0, vmax=1)
axes[2].set_title("Appartenance : région claire")
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes[1:], shrink=0.7, label="Degré d’appartenance")
plt.show()

## 5. Segmentation finale

Après, on attribue chaque pixel au cluster pour lequel il a le degré d’appartenance le plus élevé, ce qui permet d’obtenir une segmentation non floue et de la comparer à l’image de départ

In [ ]:
classes = np.argmax(U_final, axis=1).reshape(hauteur, largeur)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_array, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Image de départ")
axes[1].imshow(classes, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Segmentation finale (K = 2)")
for ax in axes:
    ax.axis("off")
plt.show()

## 6. Résultats et observations

On remarque que l’algorithme sépare les pixels selon leur intensité, avec une région plutôt sombre et une autre plus claire, aussi les cartes de chaleur gardent les degrés d’appartenance, alors que la segmentation finale choisit seulement le cluster dominant pour chaque pixel, ce qui fait perdre l’information sur les zones intermédiaires

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(np.arange(1, len(historique) + 1), historique)
plt.axhline(epsilon, color="red", linestyle="--", label="epsilon")
plt.xlabel("Itération")
plt.ylabel("Changement maximal de U")
plt.title("Évolution de la convergence")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 6. Visualisation des clusters

Ensuite, on affiche l'histogramme des niveaux de gris de l'image, puis on ajoute les centres des deux clusters obtenus avec Fuzzy C-Means pour permettre de voir comment les pixels sont répartis selon leur intensité et où se trouvent les deux centres.


In [ ]:

plt.figure(figsize=(9, 5))

# Histogramme des intensités
plt.hist(pixels.ravel(), bins=100, color="gray", alpha=0.7)

# Centres des clusters
plt.axvline(centres[0, 0], color="blue",
            linestyle="--", linewidth=2,
            label="Cluster 1 (sombre)")

plt.axvline(centres[1, 0], color="red",
            linestyle="--", linewidth=2,
            label="Cluster 2 (clair)")

plt.xlabel("Intensité des pixels")
plt.ylabel("Nombre de pixels")
plt.title("Répartition des pixels et centres des clusters")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


##7. Image Colorée
Maintenant, on utilise l’image en couleur et on garde les trois valeurs RGB de chaque pixel, puis on réduit sa taille pour limiter le temps de calcul.


In [ ]:
image_couleur = Image.open("milky-way.jpg").convert("RGB")
image_couleur = image_couleur.resize((160, 160))
image_rgb = np.asarray(image_couleur, dtype=np.float64) / 255.0
h_rgb, w_rgb, _ = image_rgb.shape
pixels_rgb = image_rgb.reshape(-1, 3)

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb)
plt.title("Image originale en couleur")
plt.axis("off")
plt.show()
print("Dimensions des pixels RGB :", pixels_rgb.shape)

##8. Application de Fuzzy C-Means sur les pixels RGB
Ensuite, on reprend la même fonction Fuzzy C-Means avec trois clusters, car chaque pixel est maintenant représenté par ses valeurs rouge, verte et bleue, et après le calcul, on range les clusters du plus sombre au plus clair pour faciliter la lecture.

In [ ]:
K_rgb = 3
centres_rgb, U_rgb, historique_rgb = fuzzy_c_means(
    pixels_rgb, K=K_rgb, m=2.0, epsilon=0.001, max_iterations=100, seed=42
)

luminosite = centres_rgb @ np.array([0.2126, 0.7152, 0.0722])
ordre_rgb = np.argsort(luminosite)
centres_rgb = centres_rgb[ordre_rgb]
U_rgb = U_rgb[:, ordre_rgb]

for j, centre in enumerate(centres_rgb):
    print(f"Cluster {j+1} - centre RGB : {np.round(centre, 3)}")
print("Nombre d'itérations :", len(historique_rgb))
print("Somme des appartenances = 1 :", np.allclose(U_rgb.sum(axis=1), 1))

##9. Cartes de chaleur et segmentation en couleur
Une fois les centres calculés, on affiche uneheatmap pour chaque cluster afin de voir où les pixels appartiennent le plus à chaque région. Ensuite, on choisit le cluster dominant pour chaque pixel et on le remplace par la couleur de son centre, ce qui donne une image segmentée.

In [ ]:
fig, axes = plt.subplots(1, K_rgb + 1, figsize=(16, 4))
axes[0].imshow(image_rgb)
axes[0].set_title("Image originale")
for j in range(K_rgb):
    carte = U_rgb[:, j].reshape(h_rgb, w_rgb)
    im = axes[j + 1].imshow(carte, cmap="magma", vmin=0, vmax=1)
    axes[j + 1].set_title(f"Appartenance au cluster {j+1}")
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes[1:], fraction=0.025, pad=0.02, label="Degré d'appartenance")
plt.show()

classes_rgb = np.argmax(U_rgb, axis=1)
segmentation_rgb = centres_rgb[classes_rgb].reshape(h_rgb, w_rgb, 3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Image de départ")
axes[1].imshow(np.clip(segmentation_rgb, 0, 1))
axes[1].set_title("Segmentation RGB (K = 3)")
for ax in axes:
    ax.axis("off")
plt.show()

##10. Visualisation des clusters RGB
On représente un échantillon de pixels selon leurs composantes rouge et bleue, puis on ajoute les centres des clusters, ca permet de voir les groupes trouvés par l’algorithme, même si la classification utilise bien les trois composantes RGB.

In [ ]:
rng = np.random.default_rng(10)
indices = rng.choice(len(pixels_rgb), size=min(4000, len(pixels_rgb)), replace=False)

plt.figure(figsize=(8, 6))
for j in range(K_rgb):
    selection = indices[classes_rgb[indices] == j]
    plt.scatter(pixels_rgb[selection, 0], pixels_rgb[selection, 2],
                s=7, alpha=0.3, label=f"Cluster {j+1}")
plt.scatter(centres_rgb[:, 0], centres_rgb[:, 2],
            c=centres_rgb, edgecolors="black", marker="X", s=220,
            label="Centres des clusters")
for j, centre in enumerate(centres_rgb):
    plt.annotate(str(j+1), (centre[0], centre[2]), xytext=(6, 5),
                 textcoords="offset points", fontsize=11)
plt.xlabel("Rouge (R)")
plt.ylabel("Bleu (B)")
plt.title("Clusters des pixels RGB (projection R-B)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, len(historique_rgb) + 1), historique_rgb)
plt.axhline(0.001, linestyle="--", color="red", label="epsilon")
plt.xlabel("Itération")
plt.ylabel("Changement maximal de U")
plt.title("Convergence pour l'image RGB")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

##Observation
Pour l'image en niveaux de gris, on remarque que les changements sont assez importants au début, avec quelques petites variations, puis ils diminuent progressivement jusqu'à ce que l'algorithme converge après 14 itérations

Pour l'image RGB, on observe aussi des variations au début, mais elles sont plus importantes et durent un peu plus longtemps, ensuite les changements diminuent jusqu'à atteindre le critère d'arrêt après 26 itérations

En comparant les deux résultats, on voit que l'image en niveaux de gris converge plus rapidement que l'image RGB, ce qui peut s'expliquer par le fait qu'on utilise seulement deux clusters et une seule valeur par pixel, alors que pour l'image RGB on utilise trois clusters et trois valeurs par pixel, mais dans les deux cas l'algorithme finit par se stabiliser

## 11. Comparaison avec un nombre de clusters différent

Pour voir l'effet du nombre de clusters, je refais la segmentation RGB avec K = 5, puis je compare le résultat avec K = 3 pour voir si les détails et les couleurs de l'image sont mieux conservés

In [ ]:
# Test avec 5 clusters sur l'image RGB
K_rgb_5 = 5
centres_rgb_5, U_rgb_5, historique_rgb_5 = fuzzy_c_means(
    pixels_rgb, K=K_rgb_5, m=2.0, epsilon=0.001,
    max_iterations=100, seed=42
)
classes_rgb_5 = np.argmax(U_rgb_5, axis=1)
segmentation_rgb_5 = centres_rgb_5[classes_rgb_5].reshape(h_rgb, w_rgb, 3)
print("Nombre d'itérations :", len(historique_rgb_5))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Image originale")
axes[1].imshow(np.clip(segmentation_rgb, 0, 1))
axes[1].set_title("Segmentation K = 3")
axes[2].imshow(np.clip(segmentation_rgb_5, 0, 1))
axes[2].set_title("Segmentation K = 5")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Avec K = 5, on distingue mieux les détails de la galaxie qu'avec K = 3, par contre l'algorithme prend plus d'itérations pour converger, 59 contre 26, donc plus de clusters permet de garder plus de détails mais demande aussi plus de calculs